# ESRI and the ArcGIS Online Environment

**Week 3 | Lecture + Lab 2: Hosted Feature Layer + POI Webmap in ArcGIS Online**

---

## Learning Objectives

By the end of this unit, you will be able to:

- Upload and publish geospatial data as a hosted feature layer in ArcGIS Online
- Configure layer properties, popups, and symbology in Map Viewer
- Build a simple point-of-interest webmap and share it with a defined audience
- Describe the cost, control, and access tradeoffs of hosted proprietary services

---

## 3.1 Hosted Feature Layers: What They Are

In Week 2, publishing meant standing up GeoServer yourself: you owned the machine, you pointed it at a shapefile or a PostGIS table, and the WMS/WFS endpoints it generated lived on infrastructure you were responsible for keeping online. ArcGIS Online (AGOL) flips that arrangement. When you publish a **hosted feature layer**, the data doesn't sit on a server you administer — it's uploaded into Esri's cloud infrastructure and stored in a managed, multi-tenant data store that Esri operates on your behalf. Esri, not you, owns the uptime, the backups, the scaling, and the patching.

This is the same publish-a-layer idea from Week 2, but with the self-hosted/hosted axis flipped, and that flip has real consequences:

- **Persistence.** A GeoServer instance keeps running for as long as you keep the server running and paying for it (or keep it running for free on your own hardware). A hosted feature layer keeps running for as long as your organization holds a valid ArcGIS Online subscription and has enough **credits** — Esri's consumption-based currency that hosted storage and services draw against.
- **Control.** With GeoServer, you can inspect the raw files, move them, back them up however you like, or migrate to a different server entirely — the data was never anywhere but where you put it. With a hosted feature layer, the canonical copy of your data lives inside Esri's platform; you interact with it through the ArcGIS REST API and Map Viewer, not through direct file access.
- **What happens if the license lapses.** If an ArcGIS Online organizational subscription expires or an account is deactivated, hosted content doesn't necessarily disappear immediately, but it does become inaccessible to the public and unusable for editing or new publishing — effectively frozen behind a paywall you no longer have access to. Contrast this with GeoServer: if you stop paying for the *server*, the underlying shapefile or database is still just a file on disk, sitting wherever it always was, fully usable with any other GIS software.

None of this makes hosted feature layers a worse choice — for a class project, a small organization without IT staff, or a map that needs to go from zero to shared-with-the-world in ten minutes, the fact that Esri runs the server for you is the entire point. But it's worth naming clearly, because it's the same tradeoff you'll see again in Week 6 with cloud-native formats: convenience and managed infrastructure in exchange for giving up direct control of where the data actually lives.

| | **GeoServer (self-hosted, Week 2)** | **ArcGIS Online (hosted feature layer)** |
|---|---|---|
| Who runs the server | You (or your organization's IT) | Esri |
| Where data physically lives | Wherever you put the shapefile/PostGIS database | Esri's managed cloud data store |
| Cost model | Free software; you pay for your own hardware/hosting | Subscription + consumable **credits** |
| If you stop paying | Server goes down, but the original files are untouched and portable | Content becomes inaccessible; you don't hold a portable local copy |
| Setup effort | You configure Store &#8594; Layer &#8594; Style by hand | Upload a file; AGOL infers schema and publishes automatically |
| Control over infrastructure | Full — you can migrate, back up, or inspect data directly | None — you interact only through Esri's interfaces and REST API |

The two aren't really competing on *capability* so much as on *who takes on the responsibility of running the server* — the same self-hosted vs. hosted tension you'll see again with cloud-native formats in Week 6.

## 3.2 Uploading and Publishing Data

The publish-from-upload workflow in ArcGIS Online is deliberately compressed compared to GeoServer's Store &#8594; Layer &#8594; Style chain from Week 2 — most of those steps still happen, but AGOL does them automatically instead of asking you to configure each one by hand.

**Formats AGOL accepts for upload:**

- **CSV** &#8212; a spreadsheet with either latitude/longitude columns, or street addresses AGOL can geocode into points during upload
- **Shapefile** &#8212; must be zipped (the .shp, .shx, .dbf, and .prj files together) before upload; AGOL will not accept the individual files loose
- **GeoJSON** &#8212; uploaded directly, geometry and attributes intact
- **File Geodatabase** (zipped) and **GPX** are also accepted, though CSV, shapefile, and GeoJSON cover most classroom use cases

**The workflow, step by step:**

1. In Map Viewer (or Content &#8594; New Item), choose **Add &#8594; Add Layer from File** and select your zipped shapefile, CSV, or GeoJSON.
2. AGOL inspects the file, infers the geometry type and field schema, and — for CSVs with address fields rather than coordinates — runs a geocoding pass to convert addresses into points, consuming credits in the process.
3. You're prompted to confirm the layer name and whether to publish it as a **hosted feature layer** (queryable, editable, symbolizable — the AGOL equivalent of a WFS-backed layer) versus adding it as a temporary file layer that only exists in this one map session.
4. Once published, AGOL generates an **Item** in your Content — this is the closest analog to GeoServer's *Layer* concept: a single published, requestable resource with its own metadata, sharing settings, and REST endpoint, generated automatically instead of configured by hand.

**Where the data actually goes:** once published, your original CSV or shapefile is no longer the live copy. AGOL has ingested it into its own hosted feature service, backed by Esri's cloud data store — the same conceptual role PostGIS plays as a *Store* in GeoServer, except you never see or manage that underlying database directly. Every edit made afterward (in Map Viewer, in the Esri field apps, or via the REST API) writes to that hosted copy, not back to your original file.

## 3.3 Configuring Popups, Symbology, and Sharing

Publishing a layer gets your data onto the map; it doesn't make that map *readable*. Two people can publish the same point-of-interest layer and end up with completely different maps depending on how they handle symbology and popups — this is the same role that an SLD file played for a WMS layer in Week 2, except here it's driven through a UI instead of hand-written XML.

**Symbology in Map Viewer.** Click a layer's *Styles* pane and you'll typically choose from:

- **Location (single symbol)** &#8212; every feature drawn identically; good for a simple POI layer where the points themselves are the message
- **Unique values** &#8212; one color/icon per distinct category in a field (e.g., a different marker for "restaurant" vs. "park" vs. "trailhead")
- **Counts and amounts (graduated colors/symbols)** &#8212; symbol size or color intensity scales with a numeric field, the standard choropleth/proportional-symbol move

This is a friendlier on-ramp than writing SLD by hand, but it's doing the same conceptual job: deciding, at the server/platform level, how a feature should look before anyone requests it.

**Popups.** By default AGOL shows every attribute field in a raw popup table — rarely what you want in a finished map. Under a layer's *Popup* settings you can pick which fields display, give them human-readable aliases (`poi_type` &#8594; "Type"), reorder them, and format numbers, dates, or URLs. This is the layer of polish that turns "here's the data" into "here's a map someone unfamiliar with your fields can actually use."

**Basemap.** Map Viewer ships with a gallery of Esri basemaps (topographic, streets, imagery, a muted "light gray canvas" good for thematic overlays). Choosing one is mostly an aesthetic/contextual decision, but it's worth remembering that most of these are themselves services published by Esri — you're consuming a hosted layer the same way an external client would consume yours.

**Sharing.** Once the map looks right, the *Share* button offers three levels, and this is where the decision stops being cosmetic and starts being consequential:

- **Private (owner only)** &#8212; visible only to you, useful while a map is still a draft
- **Organization** &#8212; visible to everyone signed into your ArcGIS Online organization, but not the open internet
- **Public (everyone)** &#8212; visible to anyone with the link, discoverable in AGOL's public search, and embeddable elsewhere

The sharing level isn't a neutral technical setting bolted on at the end — it's a design decision made at publish time, in the same way choosing WMS vs. WFS in Week 2 determined who could do what with a layer. Choosing "Public" for a map with sensitive locations (a domestic violence shelter, a culturally significant site, an at-risk species' habitat) is a different act than choosing it for a public bus-stop inventory, even though the button looks identical.

## 3.4 Who Can See This Map?

Setting a map's sharing level to "Public" feels like flipping a switch, but it's worth being precise about what actually happens: the data doesn't become public in some abstract, ownerless sense — it becomes publicly accessible *on Esri's servers, under Esri's terms of service, subject to Esri's uptime*. "Public" here means "public through a specific company's infrastructure," not "public" the way an open standard published from a self-hosted GeoServer instance is public. The map is still, at every layer, running on someone else's platform, discoverable through someone else's search index, and governed by a Terms of Service document your organization didn't write.

That distinction matters most for exactly the kind of data where the stakes are highest:

- **Community and tribal data.** Indigenous communities and other groups practicing data sovereignty may have strong reasons *not* to want culturally sensitive locations sitting on a commercial vendor's cloud, regardless of the sharing setting — the concern isn't only "can strangers see this," it's "whose infrastructure holds this, and under what terms can that access be revoked or that data be used."
- **Sensitive locations.** Habitat data for an endangered species, the address of a shelter, or precise home locations in a community survey can all be technically "correct" to map and still cause real harm if shared publicly — poaching, harassment, or exposure are real downstream risks, not hypothetical ones.
- **Persistence and revocability.** Because the data lives on Esri's platform (Section 3.1), the people who created a public map don't have unilateral control over its long-term availability — an account suspension, a lapsed license, or a platform policy change can affect access in ways a self-hosted GeoServer instance, sitting on infrastructure you control, would not.

None of this means "never share publicly" — plenty of data (bus routes, public parks, zoning boundaries) is precisely the kind of information that *should* be as open and discoverable as possible. The point is that the sharing decision deserves the same scrutiny as any other design decision in this course: who is this data about, who benefits from it being visible, and who might be put at risk by that same visibility?

**Hold onto this question** — it's the seed for Discussion 1 in Week 4, where we'll dig into open data, community consent, and what "open" actually obligates a publisher to consider.

### 5-minute discussion (breakout or whole-class)

1. Revisit Week 2's discussion question 1: who benefits from data published through an open standard, and who might still be excluded? Now flip it — who benefits from data published through a *proprietary, hosted* platform like AGOL, and who might be excluded or put at risk by that choice instead?
2. If your organization's ArcGIS Online subscription lapsed tomorrow, what would happen to a public map you'd shared widely? Compare that to what would happen to a WMS layer served from a GeoServer instance you personally administered. Which failure mode worries you more, and why?

---

## Readings & Resources

- [Esri Learn: Publish a hosted feature layer](https://learn.arcgis.com/en/projects/get-started-with-arcgis-online/)
- [ArcGIS Online: Share your map (Esri documentation)](https://doc.arcgis.com/en/arcgis-online/share-maps/share-map.htm)
- [NM RGIS data clearinghouse](https://rgis.unm.edu) — find a NM dataset to use in Lab 2

---

## Lab 2: Publishing a Hosted Feature Layer and POI Webmap

See [Lab 2](../labs/lab_02.md).